In [7]:
import os
import numpy as np
import pickle
import codecs
from collections import defaultdict
from scipy.stats import spearmanr
from tqdm import tqdm
import pickle
from collections import defaultdict
final_dict=defaultdict(list)
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import kendalltau
from contextlib import redirect_stdout

# model= Word2Vec.load("word2vec_billion.model")
folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
progress_bar = tqdm(total=dir_count, desc="Running Datasets")

for folder_name in os.listdir(folder_path):
    if folder_name == 'wordsim353-sim':
        current_folder_path = os.path.join(folder_path, folder_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, folder_name)
            dict_path = files_start_name + '_knowledge_profile_dict.pkl'
            if os.path.exists(dict_path) == False:
                progress_bar.update(1)
                continue
            saved= open(dict_path, "rb")
            profile_dict_word = pickle.load(saved)
            saved.close()

            ds_path = files_start_name + '.csv'
            if os.path.exists(ds_path) == False:
                progress_bar.update(1)
                continue
            
            fread_simlex=codecs.open(ds_path, 'r', 'utf-8')
            pair_list = []
            line_number = 0
            for line in fread_simlex:
                if line_number > 0:
                    tokens = line.split(',')
                    word_i = tokens[1].lower()
                    word_j = tokens[2].lower()
                    score = float(tokens[3].replace('\n', ''))
                    pair_list.append( ((word_i, word_j), score) )
                line_number += 1

            calculated_score=[]
            extracted_list = []
            original_score=[]
            word_pairs=[]

            
            for (x,y) in pair_list:
                if x in profile_dict_word:
                    word1, word2=x
                    word1_prof=profile_dict_word[x] * 10
                    extracted_list.append((x, word1_prof))
                    calculated_score.append(word1_prof)
                    original_score.append(y)
                    word_pairs.append(x)

            # words = (model.wv.index_to_key)
            embedding_weight_w2v=[]
            from sklearn.metrics.pairwise import cosine_similarity
            with open(files_start_name + '_evaluation.txt', 'w') as file, redirect_stdout(file):
                print(f'original score {len(original_score)} calculated score {len(calculated_score)}')

                spearman_TM = spearmanr(original_score, calculated_score)
                spearman_TM = round(spearman_TM[0], 3)
                print(f'spearman TM {spearman_TM}')

                total_list=[]
                total_list.append(original_score)
                total_list.append(calculated_score)

                similarity = cosine_similarity(total_list)
                print(f'Cosine TM {similarity}')

                TM_corr= np.corrcoef(original_score, calculated_score)
                print(f'pearson TM {TM_corr}')

                kendal_TM, _ = kendalltau(original_score, calculated_score)
                print(f'kendal TM {kendal_TM}')

                import pandas as pd
                data = pd.DataFrame([original_score,calculated_score])
                data=data.transpose()
                data.columns=['original','TM']
                correlation = data.corr()
                print("pearson corr", correlation)
        progress_bar.update(1)
progress_bar.close()

Running Datasets:   0%|          | 0/14 [06:46<?, ?it/s]

NameError: name 'pd' is not defined